# LeetCode #184 — Department Highest Salary: `JOIN`, `GROUP BY` + `MAX` và xử lý "đồng hạng"

**Mức độ:** Medium · **Chủ đề:** Database (SQL), Pandas

Skeleton trên LeetCode:

```sql
-- MySQL
# Write your MySQL query statement below
```

```python
# Pandas
import pandas as pd

def department_highest_salary(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
```

Mục tiêu của notebook:

1. **Nối hai bảng** (`JOIN … ON`) để lấy tên phòng ban từ khóa ngoại `departmentId`.
2. `GROUP BY departmentId` + `MAX(salary)` cho **mức** lương cao nhất mỗi phòng.
3. Lấy **mọi** nhân viên đạt mức đó (có thể nhiều người **đồng hạng**): lọc bằng `(departmentId, salary) IN (…)` hoặc hàm cửa sổ `RANK()`.
4. Vì sao `SELECT name, MAX(salary) … GROUP BY departmentId` là **sai**.

# 1. Bài toán là gì?

```text
Employee                                   Department
+--------------+---------+                 +-------------+---------+
| Column Name  | Type    |                 | Column Name | Type    |
+--------------+---------+                 +-------------+---------+
| id           | int     |  khóa chính     | id          | int     |  khóa chính
| name         | varchar |                 | name        | varchar |  không rỗng
| salary       | int     |                 +-------------+---------+
| departmentId | int     |  khóa ngoại → Department.id (không NULL)
+--------------+---------+
```

Tìm các nhân viên có lương **cao nhất trong phòng ban của mình**. Kết quả gồm cột `Department`, `Employee`, `Salary`, thứ tự tùy ý.

---

## Ví dụ

```text
Employee                                    Department
| id | name  | salary | departmentId |      | id | name  |
| 1  | Joe   | 70000  | 1            |      | 1  | IT    |
| 2  | Jim   | 90000  | 1            |      | 2  | Sales |
| 3  | Henry | 80000  | 2            |
| 4  | Sam   | 60000  | 2            |
| 5  | Max   | 90000  | 1            |

Kết quả
| Department | Employee | Salary |
| IT         | Jim      | 90000  |
| Sales      | Henry    | 80000  |
| IT         | Max      | 90000  |      ← Jim và Max đồng hạng, đều phải có
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. SQL cơ bản — truy vấn bảng dữ liệu
2. Hàm cửa sổ (window function) trong SQL
3. Pandas DataFrame — SQL bằng Python

Cuối mục có phần **2.4 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 SQL cơ bản — truy vấn bảng dữ liệu

### Bảng, hàng, cột

Cơ sở dữ liệu quan hệ lưu dữ liệu trong các **bảng**. Mỗi **hàng** là một bản ghi, mỗi **cột** là một thuộc tính có kiểu (số, chuỗi…). Ô có thể **rỗng** — giá trị đặc biệt `NULL` ("không biết / không có").

```text
Employee
+----+-------+--------+--------------+
| id | name  | salary | departmentId |
+----+-------+--------+--------------+
| 1  | Joe   | 70000  | 1            |
| 2  | Jim   | 90000  | 1            |
| 3  | Henry | 80000  | 2            |
+----+-------+--------+--------------+
```

Trong notebook ta chạy SQL thật bằng **`sqlite3`** (có sẵn trong Python, chạy trong bộ nhớ). LeetCode dùng MySQL — cú pháp các câu cơ bản giống nhau, chỗ khác biệt sẽ được ghi chú.

In [1]:
import sqlite3

def sql_demo(query, params=()):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE Employee (id INTEGER, name TEXT, salary INTEGER, departmentId INTEGER)")
    con.executemany("INSERT INTO Employee VALUES (?, ?, ?, ?)",
                    [(1, "Joe", 70000, 1), (2, "Jim", 90000, 1), (3, "Henry", 80000, 2),
                     (4, "Sam", 60000, 2), (5, "Max", 90000, 1)])
    con.execute("CREATE TABLE Department (id INTEGER, name TEXT)")
    con.executemany("INSERT INTO Department VALUES (?, ?)", [(1, "IT"), (2, "Sales")])
    cur = con.execute(query, params)
    cols = [d[0] for d in cur.description]
    out = cur.fetchall()
    con.close()
    print(cols)
    for r in out:
        print("  ", r)

sql_demo("SELECT * FROM Employee")

['id', 'name', 'salary', 'departmentId']
   (1, 'Joe', 70000, 1)
   (2, 'Jim', 90000, 1)
   (3, 'Henry', 80000, 2)
   (4, 'Sam', 60000, 2)
   (5, 'Max', 90000, 1)


### Thứ tự **viết** và thứ tự **thực hiện** của một câu SELECT

```text
Viết:        SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ... LIMIT ...
Thực hiện:   FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

Hiểu thứ tự thực hiện giải thích nhiều lỗi: `WHERE` chạy **trước** khi gom nhóm nên không dùng được hàm gộp (`COUNT`, `MAX`…) trong `WHERE` — phải dùng `HAVING`.

### Lọc, sắp xếp, giới hạn, bỏ trùng

In [2]:
print("Lương > 65000, giảm dần, lấy 2 hàng:")
sql_demo("SELECT name, salary FROM Employee WHERE salary > 65000 ORDER BY salary DESC LIMIT 2")
print("Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):")
sql_demo("SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1")

Lương > 65000, giảm dần, lấy 2 hàng:
['name', 'salary']
   ('Jim', 90000)
   ('Max', 90000)
Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):
['salary']
   (80000,)


### Hàm gộp và GROUP BY

`COUNT`, `SUM`, `AVG`, `MIN`, `MAX` gộp nhiều hàng thành một giá trị. `GROUP BY cột` chia bảng thành nhóm theo cột đó, gộp riêng từng nhóm. `HAVING` lọc **sau** khi gộp.

In [3]:
sql_demo("""
SELECT departmentId, COUNT(*) AS n, MAX(salary) AS top
FROM Employee
GROUP BY departmentId
HAVING COUNT(*) >= 2
""")

['departmentId', 'n', 'top']
   (1, 3, 90000)
   (2, 2, 80000)


### NULL: "không có giá trị"

- Hàm gộp trên **tập rỗng** (trừ `COUNT`) trả về `NULL` — đây là mẹo để trả về `NULL` khi "không có đáp án".
- So sánh với `NULL` bằng `=` luôn **không đúng** → phải dùng `IS NULL`.
- `IFNULL(x, d)` / `COALESCE(x, d)` thay `NULL` bằng giá trị mặc định.

In [4]:
sql_demo("SELECT MAX(salary) AS m FROM Employee WHERE salary > 1000000")
sql_demo("SELECT NULL = NULL AS bang_nhau, NULL IS NULL AS la_null, COALESCE(NULL, 0) AS mac_dinh")

['m']
   (None,)
['bang_nhau', 'la_null', 'mac_dinh']
   (None, 1, 0)


### Truy vấn con (subquery)

Một câu `SELECT` có thể nằm **bên trong** câu khác: trả về một giá trị (dùng trong `WHERE x > (...)`), một danh sách (dùng với `IN`), hoặc một bảng tạm (đặt trong `FROM`).

In [5]:
print("Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:")
sql_demo("SELECT MAX(salary) AS second FROM Employee WHERE salary < (SELECT MAX(salary) FROM Employee)")

Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:
['second']
   (80000,)


### JOIN: ghép hai bảng

`A JOIN B ON điều_kiện` tạo các cặp hàng (a, b) thỏa điều kiện.

- **INNER JOIN** (mặc định): chỉ giữ cặp khớp.
- **LEFT JOIN**: giữ **mọi** hàng của A; hàng không khớp thì cột của B là `NULL`.
- **Self-join**: ghép bảng với **chính nó** (đặt hai bí danh khác nhau) — để so sánh các hàng trong cùng bảng.

In [6]:
sql_demo("""
SELECT d.name AS department, e.name AS employee, e.salary
FROM Employee e JOIN Department d ON e.departmentId = d.id
ORDER BY d.name, e.salary DESC
""")
print("Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau")
sql_demo("""
SELECT a.name, b.name FROM Employee a JOIN Employee b
ON a.departmentId = b.departmentId AND a.salary < b.salary
""")

['department', 'employee', 'salary']
   ('IT', 'Jim', 90000)
   ('IT', 'Max', 90000)
   ('IT', 'Joe', 70000)
   ('Sales', 'Henry', 80000)
   ('Sales', 'Sam', 60000)
Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau
['name', 'name']
   ('Joe', 'Jim')
   ('Joe', 'Max')
   ('Sam', 'Henry')


### Lỗi hay gặp

- Quên `DISTINCT` khi hỏi "giá trị lớn thứ k **khác nhau**".
- Dùng hàm gộp trong `WHERE` (phải dùng `HAVING` hoặc truy vấn con).
- `LIMIT 1 OFFSET k` trên bảng **không đủ** hàng trả về **không hàng nào** (không phải `NULL`) — bọc trong truy vấn con hoặc dùng `MAX` để có `NULL`.
- So sánh `= NULL` thay vì `IS NULL`.
- Trong MySQL, `LIMIT` không nhận biểu thức (`LIMIT N-1` lỗi) → phải tính trước (`SET N = N - 1`).

## 2.2 Hàm cửa sổ (window function) trong SQL

### Hàm gộp vs hàm cửa sổ

- **Hàm gộp + GROUP BY**: mỗi nhóm **co lại** thành một hàng.
- **Hàm cửa sổ** `f(...) OVER (PARTITION BY ... ORDER BY ...)`: **giữ nguyên** mọi hàng, mỗi hàng nhận thêm một giá trị tính trên "cửa sổ" các hàng liên quan.

```text
PARTITION BY departmentId   → chia thành các nhóm (giống GROUP BY nhưng không co lại)
ORDER BY salary DESC        → thứ tự trong mỗi nhóm
```

Có từ MySQL 8.0 và SQLite 3.25.

In [7]:
import sqlite3

def window_demo(query):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE Scores (id INTEGER, score REAL, dept INTEGER)")
    con.executemany("INSERT INTO Scores VALUES (?, ?, ?)",
                    [(1, 3.50, 1), (2, 3.65, 1), (3, 4.00, 2), (4, 3.85, 2), (5, 4.00, 1), (6, 3.65, 2)])
    cur = con.execute(query)
    print([d[0] for d in cur.description])
    for r in cur.fetchall():
        print("  ", r)
    con.close()

### Ba hàm đánh số: ROW_NUMBER, RANK, DENSE_RANK

Khác nhau ở cách xử lý **giá trị bằng nhau**:

| giá trị | ROW_NUMBER | RANK | DENSE_RANK |
|---|---|---|---|
| 4.00 | 1 | 1 | 1 |
| 4.00 | 2 | 1 | 1 |
| 3.85 | 3 | **3** (nhảy) | **2** (liền) |

In [8]:
window_demo("""
SELECT score,
       ROW_NUMBER() OVER (ORDER BY score DESC) AS row_num,
       RANK()       OVER (ORDER BY score DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY score DESC) AS dense
FROM Scores ORDER BY score DESC
""")

['score', 'row_num', 'rnk', 'dense']
   (4.0, 1, 1, 1)
   (4.0, 2, 1, 1)
   (3.85, 3, 3, 2)
   (3.65, 4, 4, 3)
   (3.65, 5, 4, 3)
   (3.5, 6, 6, 4)


### PARTITION BY: đánh số riêng trong từng nhóm

"Top-k mỗi nhóm" = đánh số trong nhóm rồi lọc `<= k`. Vì `WHERE` chạy **trước** hàm cửa sổ, phải bọc trong truy vấn con hoặc `WITH`.

In [9]:
window_demo("""
WITH ranked AS (
    SELECT dept, id, score,
           DENSE_RANK() OVER (PARTITION BY dept ORDER BY score DESC) AS r
    FROM Scores
)
SELECT dept, id, score FROM ranked WHERE r = 1 ORDER BY dept, id
""")

['dept', 'id', 'score']
   (1, 5, 4.0)
   (2, 3, 4.0)


### LAG / LEAD: nhìn hàng trước / sau

`LAG(col, k)` lấy giá trị của hàng cách **trước** `k` hàng (theo `ORDER BY` của cửa sổ), `LEAD` lấy hàng **sau**; không có thì `NULL`.

In [10]:
window_demo("""
SELECT id, score,
       LAG(score, 1)  OVER (ORDER BY id) AS truoc,
       LEAD(score, 1) OVER (ORDER BY id) AS sau
FROM Scores ORDER BY id
""")

['id', 'score', 'truoc', 'sau']
   (1, 3.5, None, 3.65)
   (2, 3.65, 3.5, 4.0)
   (3, 4.0, 3.65, 3.85)
   (4, 3.85, 4.0, 4.0)
   (5, 4.0, 3.85, 3.65)
   (6, 3.65, 4.0, None)


### WITH (CTE): đặt tên cho bảng tạm

`WITH ten AS (SELECT ...) SELECT ... FROM ten` giúp chia truy vấn dài thành các bước dễ đọc — tương tự gán biến trung gian trong Python.

### Lỗi hay gặp

- Dùng `RANK` khi đề muốn thứ hạng **liền nhau** (cần `DENSE_RANK`), hoặc ngược lại.
- Lọc kết quả hàm cửa sổ trong `WHERE` cùng câu → lỗi; phải bọc truy vấn con / CTE.
- `rank` là **từ khóa** trong MySQL 8 → đặt tên cột là `` `rank` `` (backtick).
- Quên `PARTITION BY` → đánh số trên **toàn bảng** thay vì từng nhóm.

## 2.3 Pandas DataFrame — SQL bằng Python

### DataFrame là gì?

`pandas.DataFrame` là một **bảng** trong bộ nhớ: các cột có tên, mỗi cột là một `Series` (mảng có nhãn). LeetCode có phiên bản Pandas cho các bài SQL; mỗi thao tác SQL có một tương đương:

| SQL | Pandas |
|---|---|
| `WHERE cond` | `df[df["col"] > x]` |
| `SELECT a, b` | `df[["a", "b"]]` |
| `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| `DISTINCT` | `df.drop_duplicates()` / `series.unique()` |
| `GROUP BY` + gộp | `df.groupby("k")["v"].max()` |
| `JOIN` | `df1.merge(df2, left_on=..., right_on=...)` |
| `DENSE_RANK()` | `series.rank(method="dense", ascending=False)` |
| `LAG(col)` | `series.shift(1)` |

In [11]:
import pandas as pd

emp = pd.DataFrame({"id": [1, 2, 3, 4, 5],
                    "name": ["Joe", "Jim", "Henry", "Sam", "Max"],
                    "salary": [70000, 90000, 80000, 60000, 90000],
                    "departmentId": [1, 1, 2, 2, 1]})
dep = pd.DataFrame({"id": [1, 2], "name": ["IT", "Sales"]})
print(emp)

   id   name  salary  departmentId
0   1    Joe   70000             1
1   2    Jim   90000             1
2   3  Henry   80000             2
3   4    Sam   60000             2
4   5    Max   90000             1


### Lọc, chọn cột, sắp xếp, bỏ trùng

In [12]:
print(emp[emp["salary"] > 65000][["name", "salary"]].sort_values("salary", ascending=False).head(2))
print("lương khác nhau giảm dần:", sorted(emp["salary"].unique().tolist(), reverse=True))

  name  salary
1  Jim   90000
4  Max   90000
lương khác nhau giảm dần: [90000, 80000, 70000, 60000]


### groupby và transform

- `groupby(k)[v].max()` → **co** mỗi nhóm thành một giá trị (như `GROUP BY`).
- `groupby(k)[v].transform("max")` → trả về giá trị của nhóm **cho từng hàng** (giống hàm cửa sổ `MAX() OVER (PARTITION BY k)`), rất tiện để lọc "hàng có giá trị lớn nhất nhóm".

In [13]:
print(emp.groupby("departmentId")["salary"].max())
top = emp[emp["salary"] == emp.groupby("departmentId")["salary"].transform("max")]
print(top)

departmentId
1    90000
2    80000
Name: salary, dtype: int64
   id   name  salary  departmentId
1   2    Jim   90000             1
2   3  Henry   80000             2
4   5    Max   90000             1


### merge (JOIN), rank, shift

In [14]:
m = emp.merge(dep, left_on="departmentId", right_on="id", suffixes=("", "_dep"))
print(m[["name_dep", "name", "salary"]])
s = pd.Series([4.00, 4.00, 3.85, 3.65])
print("rank dense:", s.rank(method="dense", ascending=False).tolist(), "| rank min (như RANK):", s.rank(method="min", ascending=False).tolist())
print("shift(1) (như LAG):", s.shift(1).tolist())

  name_dep   name  salary
0       IT    Joe   70000
1       IT    Jim   90000
2    Sales  Henry   80000
3    Sales    Sam   60000
4       IT    Max   90000
rank dense: [1.0, 1.0, 2.0, 3.0] | rank min (như RANK): [1.0, 1.0, 3.0, 4.0]
shift(1) (như LAG): [nan, 4.0, 4.0, 3.85]


### Lỗi hay gặp

- `df[...]` lọc trả về DataFrame **có thể rỗng** — kiểm tra `len(...)` trước khi lấy `.iloc[0]`.
- `rank` trả về số thực (`1.0`) → `astype(int)` nếu cần.
- `merge` hai bảng cùng tên cột → cột bị thêm hậu tố `_x`, `_y` (hoặc `suffixes` tự đặt).
- Khi đề yêu cầu trả `null`, trả `None` trong một DataFrame một hàng — không trả DataFrame rỗng.

## 2.4 Áp dụng vào bài này

Cần **`JOIN`** hai bảng để lấy tên phòng ban, và tìm nhân viên có lương **bằng lương lớn nhất của phòng mình** — `GROUP BY departmentId` + `MAX(salary)` rồi so khớp cặp `(departmentId, salary)` (giữ **mọi** người đồng hạng), hoặc **`DENSE_RANK() OVER (PARTITION BY departmentId ...)`** rồi lọc hạng 1. Bản Pandas: `groupby(...).transform("max")` + `merge`.

# 3. Nền tảng

In [15]:
import sqlite3
import random
import pandas as pd


def run_sql(sql: str, tables: dict, params=()) -> pd.DataFrame:
    conn = sqlite3.connect(":memory:")
    for name, df in tables.items():
        df.to_sql(name, conn, index=False)
    out = pd.read_sql_query(sql, conn, params=params)
    conn.close()
    return out


def rows(df: pd.DataFrame) -> list:
    return [[None if pd.isna(v) else v for v in r] for r in df.itertuples(index=False)]


EMP = pd.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "name": ["Joe", "Jim", "Henry", "Sam", "Max"],
    "salary": [70000, 90000, 80000, 60000, 90000],
    "departmentId": [1, 1, 2, 2, 1],
})
DEP = pd.DataFrame({"id": [1, 2], "name": ["IT", "Sales"]})
T = {"Employee": EMP, "Department": DEP}

## `JOIN`: ghép dòng của hai bảng theo điều kiện

`Employee e JOIN Department d ON e.departmentId = d.id` tạo một dòng cho **mỗi cặp** (nhân viên, phòng ban) thỏa điều kiện — tức là gắn tên phòng ban vào từng nhân viên.

| Loại | Giữ dòng nào |
|---|---|
| `INNER JOIN` (mặc định `JOIN`) | chỉ các cặp khớp |
| `LEFT JOIN` | mọi dòng bảng trái; không khớp thì cột bảng phải là `NULL` |

In [16]:
print(run_sql("""
SELECT e.name AS employee, e.salary, d.name AS department
FROM Employee e
JOIN Department d ON e.departmentId = d.id
""", T))

  employee  salary department
0      Joe   70000         IT
1      Jim   90000         IT
2    Henry   80000      Sales
3      Sam   60000      Sales
4      Max   90000         IT


## `GROUP BY` + hàm tổng hợp

`GROUP BY departmentId` gộp các dòng cùng phòng thành **một** dòng; `MAX(salary)` tính trên mỗi nhóm.

In [17]:
print(run_sql("SELECT departmentId, MAX(salary) AS top FROM Employee GROUP BY departmentId", T))

   departmentId    top
0             1  90000
1             2  80000


Nhóm chỉ còn **một** dòng → không thể biết "ai" có lương đó nếu có nhiều người. Ta cần thêm một bước: tìm **mọi** nhân viên có lương bằng mức cao nhất của phòng mình.

# 4. Cách 1: lọc bằng `(departmentId, salary) IN (…)`

```sql
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee e
JOIN Department d ON e.departmentId = d.id
WHERE (e.departmentId, e.salary) IN (
    SELECT departmentId, MAX(salary)
    FROM Employee
    GROUP BY departmentId
);
```

Truy vấn con trả về các cặp `(phòng, lương cao nhất)`; mỗi nhân viên được giữ nếu cặp của họ nằm trong đó — **mọi** người đồng hạng đều được giữ.

In [18]:
SQL_IN = """
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee e
JOIN Department d ON e.departmentId = d.id
WHERE (e.departmentId, e.salary) IN (
    SELECT departmentId, MAX(salary) FROM Employee GROUP BY departmentId
)
"""
print(run_sql(SQL_IN, T))

  Department Employee  Salary
0         IT      Jim   90000
1      Sales    Henry   80000
2         IT      Max   90000


# 5. Cách 2: truy vấn con tương quan

Với mỗi nhân viên `e`, so lương với lương cao nhất **của chính phòng đó**:

```sql
WHERE e.salary = (SELECT MAX(e2.salary) FROM Employee e2 WHERE e2.departmentId = e.departmentId)
```

In [19]:
SQL_CORR = """
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee e
JOIN Department d ON e.departmentId = d.id
WHERE e.salary = (SELECT MAX(e2.salary) FROM Employee e2 WHERE e2.departmentId = e.departmentId)
"""
print(run_sql(SQL_CORR, T))

  Department Employee  Salary
0         IT      Jim   90000
1      Sales    Henry   80000
2         IT      Max   90000


# 6. Cách 3: hàm cửa sổ `RANK() OVER (PARTITION BY …)`

`PARTITION BY departmentId` xếp hạng **riêng trong từng phòng**; người có hạng 1 là cao nhất (đồng hạng thì cùng hạng 1 — `RANK` hay `DENSE_RANK` đều được ở đây vì chỉ lấy hạng 1).

In [20]:
print(run_sql("""
SELECT e.name, e.salary, e.departmentId,
       RANK() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC) AS rnk
FROM Employee e
ORDER BY e.departmentId, rnk
""", T))

    name  salary  departmentId  rnk
0    Jim   90000             1    1
1    Max   90000             1    1
2    Joe   70000             1    3
3  Henry   80000             2    1
4    Sam   60000             2    2


In [21]:
SQL_WINDOW = """
SELECT Department, Employee, Salary
FROM (
    SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary,
           RANK() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC) AS rnk
    FROM Employee e
    JOIN Department d ON e.departmentId = d.id
) AS t
WHERE rnk = 1
"""
print(run_sql(SQL_WINDOW, T))

  Department Employee  Salary
0         IT      Jim   90000
1         IT      Max   90000
2      Sales    Henry   80000


# 7. Mô phỏng bằng Python

In [22]:
def trace_department(emp, dep):
    dep_name = dict(zip(dep["id"], dep["name"]))
    top = {}
    for d, s in zip(emp["departmentId"], emp["salary"]):
        top[d] = max(top.get(d, s), s)
    print("lương cao nhất mỗi phòng:", {dep_name[d]: s for d, s in top.items()})
    out = []
    for n, s, d in zip(emp["name"], emp["salary"], emp["departmentId"]):
        keep = s == top[d]
        print(f"   {n:<6} {dep_name[d]:<6} {s:>6} {'✓ giữ' if keep else ''}")
        if keep:
            out.append([dep_name[d], n, s])
    return out

_ = trace_department(EMP, DEP)

lương cao nhất mỗi phòng:

 {'IT': 90000, 'Sales': 80000}
   Joe    IT      70000 
   Jim    IT      90000 ✓ giữ
   Henry  Sales   80000 ✓ giữ
   Sam    Sales   60000 
   Max    IT      90000 ✓ giữ


# 8. Phiên bản nộp LeetCode

**MySQL** — cách 1 (mục 4). **Pandas:** `groupby(...).transform("max")` gắn mức lương cao nhất của phòng vào **từng dòng**, rồi lọc và nối với `Department`.

In [23]:
def department_highest_salary(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    top = employee.groupby("departmentId")["salary"].transform("max")
    best = employee[employee["salary"] == top]
    merged = best.merge(department, left_on="departmentId", right_on="id", suffixes=("_emp", "_dep"))
    return merged.rename(columns={"name_dep": "Department", "name_emp": "Employee", "salary": "Salary"})[
        ["Department", "Employee", "Salary"]]

print(department_highest_salary(EMP, DEP))

  Department Employee  Salary
0         IT      Jim   90000
1      Sales    Henry   80000
2         IT      Max   90000


In [24]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<20} PASS")


def as_sorted(df):
    return sorted(rows(df))


EXP = sorted([["IT", "Jim", 90000], ["Sales", "Henry", 80000], ["IT", "Max", 90000]])
check("Ví dụ (IN)", as_sorted(run_sql(SQL_IN, T)), EXP)
check("Ví dụ (tương quan)", as_sorted(run_sql(SQL_CORR, T)), EXP)
check("Ví dụ (RANK)", as_sorted(run_sql(SQL_WINDOW, T)), EXP)
check("Ví dụ (Pandas)", as_sorted(department_highest_salary(EMP, DEP)), EXP)
check("Tên cột", list(run_sql(SQL_IN, T).columns), ["Department", "Employee", "Salary"])


def random_tables(n_emp, n_dep):
    dep = pd.DataFrame({"id": range(1, n_dep + 1), "name": [f"D{i}" for i in range(1, n_dep + 1)]})
    emp = pd.DataFrame({
        "id": range(1, n_emp + 1),
        "name": [f"E{i}" for i in range(1, n_emp + 1)],
        "salary": [random.randint(1, 4) * 1000 for _ in range(n_emp)],
        "departmentId": [random.randint(1, n_dep) for _ in range(n_emp)],
    })
    return emp, dep


random.seed(184)
for _ in range(300):
    emp, dep = random_tables(random.randint(1, 10), random.randint(1, 3))
    top = {}
    for d, s in zip(emp["departmentId"], emp["salary"]):
        top[d] = max(top.get(d, s), s)
    exp = sorted([[f"D{d}", n, s] for n, s, d in zip(emp["name"], emp["salary"], emp["departmentId"]) if s == top[d]])
    t = {"Employee": emp, "Department": dep}
    got = [as_sorted(run_sql(q, t)) for q in (SQL_IN, SQL_CORR, SQL_WINDOW)] + [as_sorted(department_highest_salary(emp, dep))]
    assert all(g == exp for g in got), (emp, dep)
print("300 bộ bảng ngẫu nhiên: ba câu SQL và bản Pandas khớp")

Ví dụ (IN)           PASS


Ví dụ (tương quan)   PASS
Ví dụ (RANK)         PASS
Ví dụ (Pandas)       PASS


Tên cột              PASS


300 bộ bảng ngẫu nhiên: ba câu SQL và bản Pandas khớp


# 9. Vì sao đúng? Độ phức tạp?

Một nhân viên thuộc đáp án ⟺ lương của họ bằng `MAX(salary)` của phòng họ. Cả ba cách đều kiểm tra đúng điều kiện này: cách 1 qua tập cặp `(phòng, max)`, cách 2 qua truy vấn con tính `max` của phòng tương ứng, cách 3 qua hạng 1 trong phòng (hạng 1 ⟺ không ai lớn hơn ⟺ bằng max).

| Cách | Độ phức tạp điển hình |
|---|---|
| `IN` + `GROUP BY` | `O(n)` – `O(n log n)` (một lần gộp + một lần lọc) |
| Truy vấn con tương quan | `O(n²)` nếu không có chỉ mục |
| `RANK()` | `O(n log n)` |

# 10. Những lỗi dễ mắc

## Lỗi 1: `SELECT name, MAX(salary) … GROUP BY departmentId`

In [25]:
SQL_BARE = """
SELECT d.name AS Department, e.name AS Employee, MAX(e.salary) AS Salary
FROM Employee e JOIN Department d ON e.departmentId = d.id
GROUP BY e.departmentId
"""
print(run_sql(SQL_BARE, T))

  Department Employee  Salary
0         IT      Jim   90000
1      Sales    Henry   80000


Mỗi phòng chỉ còn **một** dòng → mất `Max` (đồng hạng với `Jim`). Tệ hơn, cột `name` không nằm trong `GROUP BY` và không phải hàm tổng hợp: MySQL (chế độ `ONLY_FULL_GROUP_BY` mặc định) **báo lỗi**; một số hệ khác trả về tên của một dòng **bất kỳ** trong nhóm. (SQLite có quy tắc riêng: với `MAX`, cột "trần" lấy từ dòng đạt max — nhưng vẫn chỉ một dòng.)

## Lỗi 2: So lương với **danh sách** các mức cao nhất mà không kèm phòng

In [26]:
SQL_GLOBAL = """
SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary
FROM Employee e JOIN Department d ON e.departmentId = d.id
WHERE e.salary IN (SELECT MAX(salary) FROM Employee GROUP BY departmentId)
"""
emp_bug3 = pd.DataFrame({"id": [1, 2, 3], "name": ["A", "B", "C"], "salary": [60000, 90000, 60000], "departmentId": [1, 2, 2]})
tb3 = {"Employee": emp_bug3, "Department": pd.DataFrame({"id": [1, 2], "name": ["IT", "Sales"]})}
print("IT: A=60000 (max của IT) | Sales: B=90000 (max), C=60000")
print("Sai :", as_sorted(run_sql(SQL_GLOBAL, tb3)))
print("Đúng:", as_sorted(run_sql(SQL_IN, tb3)))

IT: A=60000 (max của IT) | Sales: B=90000 (max), C=60000
Sai : [['IT', 'A', 60000], ['Sales', 'B', 90000], ['Sales', 'C', 60000]]
Đúng: [['IT', 'A', 60000], ['Sales', 'B', 90000]]


`salary IN (danh sách các max)` không gắn mức lương với phòng: `C` (60000, phòng Sales) bị giữ nhầm vì 60000 là max của **IT**. Phải so **cặp** `(departmentId, salary)`.

## Lỗi 3: `LIMIT 1` / `ROW_NUMBER()` — bỏ mất người đồng hạng

In [27]:
SQL_ROWNUM = SQL_WINDOW.replace("RANK()", "ROW_NUMBER()")
print("Sai :", as_sorted(run_sql(SQL_ROWNUM, T)))
print("Đúng:", as_sorted(run_sql(SQL_WINDOW, T)))

Sai : [['IT', 'Jim', 90000], ['Sales', 'Henry', 80000]]
Đúng: [['IT', 'Jim', 90000], ['IT', 'Max', 90000], ['Sales', 'Henry', 80000]]


# 11. Mẹo nhận diện pattern

- **"Dòng có giá trị lớn nhất trong mỗi nhóm"** (kèm thông tin khác của dòng) → `(nhóm, giá trị) IN (SELECT nhóm, MAX(...) GROUP BY nhóm)` hoặc `RANK() OVER (PARTITION BY nhóm ...) = 1`.
- **Đồng hạng phải giữ hết** → `RANK`/`DENSE_RANK`, không dùng `ROW_NUMBER`/`LIMIT`.
- **Top `k` mỗi nhóm** → `DENSE_RANK() <= k` (bài tập 4, bài #185).

Bài liên quan: **#185 Department Top Three Salaries** (Hard), **#176**, **#178**, **#1077 Project Employees III**, **#586**, **#1112 Highest Grade For Each Student**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với bảng `EMP`, `DEP` ở mục 3 nhưng đổi lương của `Sam` thành `80000`, kết quả là gì?

## Bài 2 — Viết câu SQL cách `IN`

In [28]:
def exercise_2_sql() -> str:
    # TODO
    pass


# assert sorted(rows(run_sql(exercise_2_sql(), {"Employee": EMP, "Department": DEP}))) == sorted([["IT", "Jim", 90000], ["Sales", "Henry", 80000], ["IT", "Max", 90000]])

## Bài 3 — Viết câu SQL cách hàm cửa sổ

In [29]:
def exercise_3_sql() -> str:
    # TODO
    pass


# assert sorted(rows(run_sql(exercise_3_sql(), {"Employee": EMP, "Department": DEP}))) == sorted([["IT", "Jim", 90000], ["Sales", "Henry", 80000], ["IT", "Max", 90000]])

## Bài 4 — Biến thể: hai **mức** lương cao nhất mỗi phòng (giống #185 với 2 thay 3)

Trả về `Department, Employee, Salary` của mọi người có lương thuộc **hai mức cao nhất khác nhau** trong phòng.

In [30]:
EMP4 = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "name": ["Joe", "Jim", "Henry", "Sam", "Max", "Ann"],
    "salary": [70000, 90000, 80000, 60000, 90000, 50000],
    "departmentId": [1, 1, 2, 2, 1, 1],
})

def exercise_4_sql() -> str:
    # TODO
    pass


# assert sorted(rows(run_sql(exercise_4_sql(), {"Employee": EMP4, "Department": DEP}))) == sorted([["IT", "Jim", 90000], ["IT", "Max", 90000], ["IT", "Joe", 70000], ["Sales", "Henry", 80000], ["Sales", "Sam", 60000]])

## Bài 5 — Mini challenge: bản Pandas cho bài 4

In [31]:
def exercise_5_top2_pandas(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    # TODO: trả về DataFrame cột Department, Employee, Salary
    pass


# assert sorted(rows(exercise_5_top2_pandas(EMP4, DEP))) == sorted([["IT", "Jim", 90000], ["IT", "Max", 90000], ["IT", "Joe", 70000], ["Sales", "Henry", 80000], ["Sales", "Sam", 60000]])

# 13. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_department` với bảng đã sửa.

**Bài 4**
- Mức 1: `DENSE_RANK() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC) AS dr` trong truy vấn con; lọc `dr <= 2`.

**Bài 5**
- Mức 1: `employee.groupby("departmentId")["salary"].rank(method="dense", ascending=False)`; lọc `<= 2`; `merge` với `department`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [32]:
emp1 = EMP.copy()
emp1.loc[emp1["name"] == "Sam", "salary"] = 80000
print(trace_department(emp1, DEP))

lương cao nhất mỗi phòng: {'IT': 90000, 'Sales': 80000}
   Joe    IT      70000 
   Jim    IT      90000 ✓ giữ
   Henry  Sales   80000 ✓ giữ
   Sam    Sales   80000 ✓ giữ
   Max    IT      90000 ✓ giữ
[['IT', 'Jim', 90000], ['Sales', 'Henry', 80000], ['Sales', 'Sam', 80000], ['IT', 'Max', 90000]]


## Đáp án Bài 2

In [33]:
def answer_2_sql() -> str:
    return SQL_IN

assert sorted(rows(run_sql(answer_2_sql(), {"Employee": EMP, "Department": DEP}))) == sorted([["IT", "Jim", 90000], ["Sales", "Henry", 80000], ["IT", "Max", 90000]])
print("Bài 2 PASS (giống SQL_IN ở mục 4)")

Bài 2 PASS (giống SQL_IN ở mục 4)


## Đáp án Bài 3

In [34]:
def answer_3_sql() -> str:
    return SQL_WINDOW

assert sorted(rows(run_sql(answer_3_sql(), {"Employee": EMP, "Department": DEP}))) == sorted([["IT", "Jim", 90000], ["Sales", "Henry", 80000], ["IT", "Max", 90000]])
print("Bài 3 PASS (giống SQL_WINDOW ở mục 6)")

Bài 3 PASS (giống SQL_WINDOW ở mục 6)


## Đáp án Bài 4

In [35]:
def answer_4_sql() -> str:
    return """
    SELECT Department, Employee, Salary
    FROM (
        SELECT d.name AS Department, e.name AS Employee, e.salary AS Salary,
               DENSE_RANK() OVER (PARTITION BY e.departmentId ORDER BY e.salary DESC) AS dr
        FROM Employee e JOIN Department d ON e.departmentId = d.id
    ) AS t
    WHERE dr <= 2
    """

assert sorted(rows(run_sql(answer_4_sql(), {"Employee": EMP4, "Department": DEP}))) == sorted([["IT", "Jim", 90000], ["IT", "Max", 90000], ["IT", "Joe", 70000], ["Sales", "Henry", 80000], ["Sales", "Sam", 60000]])
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [36]:
def answer_5_top2_pandas(employee: pd.DataFrame, department: pd.DataFrame) -> pd.DataFrame:
    r = employee.groupby("departmentId")["salary"].rank(method="dense", ascending=False)
    best = employee[r <= 2]
    merged = best.merge(department, left_on="departmentId", right_on="id", suffixes=("_emp", "_dep"))
    return merged.rename(columns={"name_dep": "Department", "name_emp": "Employee", "salary": "Salary"})[
        ["Department", "Employee", "Salary"]]

assert sorted(rows(answer_5_top2_pandas(EMP4, DEP))) == sorted([["IT", "Jim", 90000], ["IT", "Max", 90000], ["IT", "Joe", 70000], ["Sales", "Henry", 80000], ["Sales", "Sam", 60000]])
for _ in range(200):
    emp, dep = random_tables(random.randint(1, 10), random.randint(1, 3))
    t = {"Employee": emp, "Department": dep}
    assert sorted(rows(answer_5_top2_pandas(emp, dep))) == sorted(rows(run_sql(answer_4_sql(), t)))
print("Bài 5 PASS (khớp câu SQL bài 4 trên 200 bảng ngẫu nhiên)")

Bài 5 PASS (khớp câu SQL bài 4 trên 200 bảng ngẫu nhiên)


# 15. Tổng kết

> Nối `Employee` với `Department` để có tên phòng; giữ nhân viên có `(departmentId, salary)` thuộc tập `(departmentId, MAX(salary))` của từng phòng — hoặc `RANK() OVER (PARTITION BY departmentId ORDER BY salary DESC) = 1`. Luôn giữ **mọi** người đồng hạng; không chọn cột không gộp kèm `GROUP BY`.

---

## Checklist trước khi nộp

- [ ] So **cặp** (phòng, lương), không chỉ lương?
- [ ] Giữ mọi người đồng hạng (không `LIMIT 1`, không `ROW_NUMBER`)?
- [ ] Tên cột `Department`, `Employee`, `Salary`?
- [ ] Không dùng cột "trần" kèm `GROUP BY`?